# Advvideo4you 2.1 — AI ad video from one product photo

Turns a product photo, a name, a language (Hindi, English or Bangla) and a call to action into a **15-second 9:16 MP4** (480×832) with AI motion, an AI-written script, voice-over, animated captions, background music and a closing call-to-action card. Runs on the **free Colab T4**.

## Five steps

1. On the website press **Start Free**, choose your photo, fill in the form. It saves `advvideo-launch.json` (your settings and the photo) and opens this notebook.
2. **Open Colab** (you are here). Check that `Runtime -> Change runtime type` says **T4 GPU**.
3. **Run all** (`Runtime -> Run all`).
4. **Upload the photo**: when the upload button appears, choose `advvideo-launch.json`. That is the only click. (No launch file? Upload just a product photo instead; the notebook asks for the name, language and CTA right here.)
5. **Download the video**: the finished MP4 downloads automatically.

**Estimated time:** first run 15–25 minutes (the models download once), later runs 5–8 minutes. These are estimates; they have not been measured on a real T4 by the author.

## What happens

1. **Script.** A small local LLM (Qwen2.5-1.5B-Instruct, Apache-2.0) writes a hook and three feature lines. The output is checked (length, correct script, no numbers or risky claims); if it fails, curated templates are used. Your CTA is always used exactly as you wrote it.
2. **Voice-over.** Piper TTS, sped up slightly if needed so everything fits in 15 seconds.
3. **Video.** Wan2.1 VACE-1.3B via the official repository: your photo is frame 0, then 33 frames (about 2 seconds) at 18 steps in fp16 (bf16 only if fp16 overflows; a shorter clip only if the GPU runs out of memory). The clip is played forward and backward in a seamless loop with a slow zoom and a slight pan, and fades in and out, to fill the 15 seconds. The AI motion repeats; the voice, captions and music carry the ad. The audio work runs while the GPU is busy.
4. **Captions.** Timed to the voice, on rounded dark pills with a shadow, above the bottom 20% of the screen. Drawn by FFmpeg (libass, which also shapes Hindi and Bangla correctly; `drawtext` is the fallback). An SRT file is saved too.
5. **Closing card.** For the last 3 seconds: the product name, your CTA (default "Order Now") and "WhatsApp Today", animated.
6. **Music.** An original synthesised loop (CC0) at about 20% of the voice level, fading in and out and ducking a little more under the voice.
7. **Render.** FFmpeg merges everything to H.264 + AAC and checks the result.

## Timing and caching

The first run downloads about 19 GB of Wan files plus about 3 GB for the script model and voices. That download starts as soon as you have uploaded your file and runs while the packages install. Files are kept in `/content` for the rest of the session, and the encoded prompt is cached too, so later runs skip all of it. Turn on `CACHE_TO_DRIVE` in the cell to keep the cache in your Google Drive between sessions (Colab asks you to authorise Drive once).

## Licences and credits

- **Hindi voice-over: Piper `hi_IN-priyamvada-medium`, CC BY-NC-SA 4.0 — non-commercial use only.** English voice: `en_US-ljspeech-medium` (public domain). Bangla voice: `bn_BD-google-medium` (CC BY-SA 4.0, OpenSLR 37, with CMU terms; keep the attribution).
- Video model: Wan2.1 VACE-1.3B (Apache-2.0). Script model: Qwen2.5-1.5B-Instruct (Apache-2.0). Fonts: Noto Sans (SIL OFL). Music: original, CC0. Piper is GPL-3.0 and is installed as a separate tool.
- Do not present AI-generated lines as factual product claims; check the script before publishing.

In [ ]:
#@title Advvideo4you 2.1 - AI ad video from one photo  (Runtime > Run all)
# One cell does everything. It asks for ONE upload (the launch file from the website, or just a photo),
# then writes the script, records the voice-over, makes the AI clip, adds captions and music, renders the MP4
# and downloads it. Nothing needs editing and no runtime restart is needed.
import os
import shutil
import subprocess
import sys
import threading
import time

MUSIC = "upbeat"  #@param ["upbeat", "calm", "none"]
SCRIPT_MODE = "auto"  #@param ["auto", "templates"]
FRAMES = 33  #@param {type:"integer"}
STEPS = 18  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
CACHE_TO_DRIVE = False  #@param {type:"boolean"}

REPO_URL = "https://github.com/bnkgroups022-cloud/Advvideo4you.git"
REPO_DIR = "/content/Advvideo4you"
INPUT_DIR = "/content/advvideo_input"
WORK_DIR = "/content/advvideo_work"
OUT_DIR = "/content/advvideo_output"
CACHE_DIR = "/content/advvideo_cache"

CHECK_CODE = r'''
import torch, transformers, diffusers, accelerate, easydict, einops, ftfy, regex, imageio, imageio_ffmpeg, piper, numpy, PIL
from diffusers.models.modeling_utils import ModelMixin
from diffusers.schedulers.scheduling_utils import KarrasDiffusionSchedulers, SchedulerMixin, SchedulerOutput
from transformers import AutoModelForCausalLM, AutoTokenizer
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| diffusers", diffusers.__version__,
      "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "torch cannot see the GPU"
'''


def stamp(msg):
    print(time.strftime("[%H:%M:%S] ") + msg, flush=True)


def run_streamed(cmd, env=None, cwd=None):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env, cwd=cwd)
    for line in proc.stdout:
        print(line, end="", flush=True)
    return proc.wait()


def run_captured(cmd, cwd=None):
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd)
    return proc.returncode, proc.stdout.strip()


def main():
    started = time.time()
    stamp("Step 1/5  checking the runtime")
    code, smi = run_captured(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
    if code != 0 or not smi:
        raise SystemExit("No GPU found. Use Runtime > Change runtime type > T4 GPU, then Runtime > Run all.")
    print("GPU:", smi.splitlines()[0])
    if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
        stamp("installing ffmpeg")
        run_streamed(["apt-get", "-qq", "install", "-y", "ffmpeg"])
    if shutil.which("ffmpeg") is None:
        raise SystemExit("ffmpeg is missing and could not be installed.")
    if shutil.disk_usage("/content").free < 30 * 10 ** 9 and not os.path.isdir(os.path.join(CACHE_DIR, "Wan2.1-VACE-1.3B")):
        raise SystemExit("Not enough free disk (about 30 GB needed). Runtime > Disconnect and delete runtime, then Run all.")

    stamp("Step 2/5  getting the Advvideo4you code from GitHub")
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        run_streamed(["git", "-C", REPO_DIR, "pull", "-q", "--ff-only"])
    else:
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        if run_streamed(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR]) != 0:
            raise SystemExit("Could not clone " + REPO_URL)
    sys.path.insert(0, REPO_DIR)
    from advvideo import __version__, config
    print("Advvideo4you", __version__)

    stamp("Step 3/5  upload: choose advvideo-launch.json from the website (or just a product photo)")
    os.makedirs(INPUT_DIR, exist_ok=True)
    uploaded = {}
    try:
        from google.colab import files
        uploaded = files.upload()
    except ImportError:
        pass
    found = config.classify_uploads(uploaded)
    if found["ignored"]:
        print("ignoring:", ", ".join(found["ignored"]))
    launch_path = os.path.join(INPUT_DIR, "advvideo-launch.json")
    image_path = None
    if found["image"]:
        image_path = os.path.join(INPUT_DIR, "uploaded_" + os.path.basename(found["image"][0]))
        with open(image_path, "wb") as fh:
            fh.write(found["image"][1])
    if found["launch"]:
        with open(launch_path, "wb") as fh:
            fh.write(found["launch"])
    elif image_path:
        print("No launch file uploaded - answer three quick questions (press Enter to accept a default).")
        cfg = config.prompt_settings()
        with open(image_path, "rb") as fh:
            blob = fh.read()
        with open(launch_path, "w", encoding="utf-8") as fh:
            fh.write(config.build_launch(cfg["product_name"], cfg["language"], cfg["cta"], blob))
        image_path = None
    else:
        raise SystemExit("Nothing usable was uploaded. Run the cell again and choose advvideo-launch.json (from the website) or a product photo.")

    cache = CACHE_DIR
    if CACHE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        cache = "/content/drive/MyDrive/advvideo4you_cache"
    os.makedirs(cache, exist_ok=True)

    # The big downloads (script model, Wan2.1 weights) start now and run while the packages install; the pipeline waits
    # for them only if they are still running when it needs them. On a warm cache this returns almost at once.
    prefetch = subprocess.Popen([sys.executable, "-m", "advvideo.pipeline", "--prefetch", "--cache", cache, "--work", WORK_DIR, "--out", OUT_DIR],
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=REPO_DIR,
                                env=dict(os.environ, PYTHONPATH=REPO_DIR, PYTHONUNBUFFERED="1"))

    def show_prefetch():
        for line in prefetch.stdout:
            print(line, end="", flush=True)

    threading.Thread(target=show_prefetch, daemon=True).start()

    stamp("Step 4/5  installing packages while the models download in the background (no restart needed)")
    pip = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check"]
    plans = [
        ["easydict", "einops", "ftfy", "regex", "imageio", "imageio-ffmpeg", "diffusers", "piper-tts", "accelerate", "sentencepiece", "protobuf"],
        ["-U", "diffusers", "transformers"],
    ]
    ready = False
    for extra in plans:
        if run_streamed(pip + extra) != 0:
            print("pip step failed; trying the next option")
            continue
        code, out = run_captured([sys.executable, "-c", CHECK_CODE])
        print(out.splitlines()[-1] if out else "")
        if code == 0:
            ready = True
            break
    if not ready:
        raise SystemExit("Packages could not be made importable. The last message is printed above.")

    stamp("Step 5/5  making the ad (script, voice, video, captions, music, render)")
    shutil.rmtree(OUT_DIR, ignore_errors=True)
    cmd = [sys.executable, "-m", "advvideo.pipeline", "--launch", launch_path, "--work", WORK_DIR, "--cache", cache, "--out", OUT_DIR,
           "--music", MUSIC, "--script-mode", SCRIPT_MODE, "--frames", str(FRAMES), "--steps", str(STEPS), "--seed", str(SEED)]
    if image_path:
        cmd += ["--image", image_path]
    env = dict(os.environ, PYTHONPATH=REPO_DIR, PYTHONUNBUFFERED="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    code = run_streamed(cmd, env=env, cwd=REPO_DIR)
    videos = sorted(f for f in os.listdir(OUT_DIR) if f.endswith(".mp4")) if os.path.isdir(OUT_DIR) else []
    if code == 64:
        raise SystemExit("The input was not usable (see the message above).")
    if not videos:
        raise SystemExit("No video was produced (exit code %s). The messages above say which step failed." % code)
    final = os.path.join(OUT_DIR, videos[0])
    stamp("done in %.1f min: %s" % ((time.time() - started) / 60.0, final))
    print("Captions file:", os.path.join(OUT_DIR, "captions.srt"), "| script:", os.path.join(OUT_DIR, "script.json"))
    report_path = os.path.join(OUT_DIR, "report.json")
    if os.path.exists(report_path):
        import json
        with open(report_path, "r", encoding="utf-8") as fh:
            report = json.load(fh)
        print("Motion:", report["motion"])
        print("Voice:", report["voice"], "-", report["voice_license"])
        if report["problems"]:
            print("WARNING - output check:", "; ".join(report["problems"]))
    try:
        from IPython.display import Video, display
        display(Video(final, embed=True, width=270))
    except Exception:
        pass
    try:
        from google.colab import files
        files.download(final)
    except ImportError:
        print("saved to", final)


main()